# Data exploration
 This first notebook is used to understand better the dataset. The goal is to study the features, the distribution of the dataset and the framework.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "src" / "data_preprocessing.py").is_file()
     and (path / "notebooks").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Start Jupyter from the project folder or a notebook subfolder.")


First, let us upload the dataset. It is already split in train set, validation set and test set. 

In [ ]:
from pathlib import Path

data_dir = PROJECT_ROOT / "data"
if not all((data_dir / name).is_file() for name in (
    "train_data.csv", "val_data.csv", "test_data.csv"
)):
    raise FileNotFoundError("Dataset missing: see data/README.md for the required CSV files.")


train_data = pd.read_csv(data_dir / "train_data.csv", header=None)
val_data = pd.read_csv(data_dir / "val_data.csv", header=None)
test_data = pd.read_csv(data_dir / "test_data.csv", header=None)

# Let's have a look at the dimensions of the sets.
print("Training set shape:", train_data.shape)
print("Validation set shape:", val_data.shape)
print("Test set shape:", test_data.shape)

# Let's see the first 5 rows of the training set:
train_data.head()

The first 250 columns are the ECG samples. The last column is the label of the sample. It's represented as a float number, so we need to convert it into integer, or directly into the right label.

Now we need to separate the samples from the label:


In [ ]:
# Separate features and labels

X_train = train_data.iloc[:, :-1].to_numpy()
y_train = train_data.iloc[:, -1].astype(int).to_numpy()

X_val = val_data.iloc[:, :-1].to_numpy()
y_val = val_data.iloc[:, -1].astype(int).to_numpy()

X_test = test_data.iloc[:, :-1].to_numpy()
y_test = test_data.iloc[:, -1].astype(int).to_numpy()

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

print("X_val shape:", X_val.shape)
print("y_val shape:", y_val.shape)

print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

We obtained a numpy array that follows the usual convention so that each row represents a sample and each columns is a feature. The y vector is a monodimensional vector containing the labels.

Now we can check the quality of the dataset, assessing the number of missing values, infinite values and computing some metrics on the dataset. 
It has to be noticed that the dataset has already been pre-processed. In any case, it is part of the dataset exploration to check the following points.

In [ ]:
print("Missing values in training set:", np.isnan(X_train).sum())
print("Infinite values in training set:", np.isinf(X_train).sum())

print("Minimum ECG value:", X_train.min())
print("Maximum ECG value:", X_train.max())
print("Mean ECG value:", X_train.mean())
print("Standard deviation:", X_train.std())

As we expected, the dataset is already clean. The professor said that the classes are heavily unbalanced, so let's verify it.

In [ ]:
class_names = {
    0: "N",
    1: "/",
    2: "L",
    3: "R",
    4: "e",
    5: "j",
    6: "A",
    7: "a",
    8: "J",
    9: "S",
    10: "E",
    11: "F",
    12: "V",
    13: "f",
    14: "Q"
}

train_counts = np.bincount(y_train, minlength=15)
val_counts = np.bincount(y_val, minlength=15)
test_counts = np.bincount(y_test, minlength=15)

distribution = pd.DataFrame({
    "Class": np.arange(15),
    "Label": [class_names[i] for i in range(15)],
    "Train": train_counts,
    "Validation": val_counts,
    "Test": test_counts
})

display(distribution)

labels = [f"{i} ({class_names[i]})" for i in range(15)]

plt.figure(figsize=(12, 5))
plt.bar(labels, train_counts)

plt.xlabel("Class")
plt.ylabel("Number of samples")
plt.title("Class distribution in the training set")
plt.grid(axis="y", alpha=0.3)

plt.show()

As we can see, the classes are heavily unbalanced. Class 9 has only two occurences in the training set and none in the validation and test sets! It means that we can't assess anything about the generalization performance of any model on this class. Also classes 4 and 14 have really few samples, and in general most of the samples are of class 0. 

It is interesting to notice that accuracy has to be taken carefully as a metric for this problem. Indeed, a model that always classify a sample as class 0 has an accuracy of 68%, which seems acceptable.

Let's now display some examples for each class. We will display first the first heartbeat for each class, and then the first 5 to see the overlapping.

In [ ]:
# Plot one ECG beat from each class
classes, counts = np.unique(y_train, return_counts=True)
percentages = 100 * counts / len(y_train)

fig, axes = plt.subplots(3, 5, figsize=(15, 9))
axes = axes.flatten()

for cls, ax in zip(classes, axes):
    idx = np.where(y_train == cls)[0][0]

    ax.plot(X_train[idx])
    ax.set_title(f"Class {cls} ({class_names[cls]})")
    ax.set_xlabel("Sample")
    ax.set_ylabel("Amplitude")
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

Since beat segmentation has already been applied, the main event for each class appears approximately at the same sample isntant. The differences between classes are in the amplitude, baseline level and waveform shape. However, we can't say that one example per class is representative for that class.


In [ ]:
# Plot multiple ECG beats from each class

fig, axes = plt.subplots(5, 3, figsize=(15, 10))
axes = axes.flatten()

for cls, ax in zip(range(15), axes):
    indices = np.where(y_train == cls)[0][:5]

    for idx in indices:
        ax.plot(X_train[idx], alpha=0.7)

    ax.set_title(f"Class {cls} ({class_names[cls]})")
    ax.set_xlabel("Sample")
    ax.set_ylabel("Amplitude")
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

We can notice that the beats are generally aligned around the same central event, which is consistent with the segmentation already applied to the dataset. However, we can notice some differences in baseline, amplitude and waveform shape within the same class.

Initially, we will train a model that classify an ECG into 5 classes (AAMI standard), not into 15. It is then better to add some data_exploration also for that classification problem.

The grouping is done with a function group_aami_classes() in the data_preprocessing script. The map is the following:
0: N  -> N
1: /  -> Q
2: L  -> N
3: R  -> N
4: e  -> N
5: j  -> N
6: A  -> S
7: a  -> S
8: J  -> S
9: S  -> S
10: E -> V
11: F -> F
12: V -> V
13: f -> Q
14: Q -> Q

In [ ]:
import sys
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from data_preprocessing import group_aami_classes

y_train_aami = group_aami_classes(y_train)
y_val_aami = group_aami_classes(y_val)
y_test_aami = group_aami_classes(y_test)

aami_names = {0: "N", 1: "S", 2: "V", 3: "F", 4: "Q"}

aami_train_counts = np.bincount(y_train_aami, minlength=5)
aami_val_counts = np.bincount(y_val_aami, minlength=5)
aami_test_counts = np.bincount(y_test_aami, minlength=5)

aami_distribution = pd.DataFrame({
    "Class": np.arange(5),
    "Label": [aami_names[i] for i in range(5)],
    "Train": aami_train_counts,
    "Validation": aami_val_counts,
    "Test": aami_test_counts,
    "Train %": 100 * aami_train_counts / len(y_train_aami),
})

display(aami_distribution)

This 5-class classification problem is way easier than the original one, because even though the classes are unbalanced, each class has many occurrencies. 
However, it is still necessary to choose properly the metrics because accuracy is not representative of the real performance of the model.